In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import xgboost as xgb
import imblearn

print(f"TensorFlow version: {tf.__version__}")
print(f"Imbalanced-Learn version: {imblearn.__version__}")

TensorFlow version: 2.20.0
Imbalanced-Learn version: 0.14.1


In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import LabelEncoder

print("=== Phase 3: Commencing Preprocessing Layer ===")

# Define the exact official 49-column schema for the raw UNSW-NB15 files
col_names = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss',
    'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz',
    'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports',
    'ct_src_ltm', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login',
    'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm_d', 'ct_srv_dst', 'ct_state_ttl', 'ct_src_user_ltm',
    'ct_src_zone_ltm', 'ct_dst_host_ltm', 'ct_srv_src', 'ct_dst_sport_ltm_d', 'ct_dst_src_ltm_d',
    'ct_src_ltm_d_d', 'ct_src_ltm_d_s', 'ct_dst_ltm_d_d', 'ct_dst_ltm_d_s', 'ct_srv_dst_d', 'ct_srv_src_d',
    'ct_state_ttl_d', 'ct_src_user_ltm_d', 'ct_src_zone_ltm_d', 'ct_dst_host_ltm_d', 'ct_srv_dst_d_d',
    'ct_srv_src_d_d', 'ct_state_ttl_d_d', 'ct_src_user_ltm_d_d', 'ct_src_zone_ltm_d_d', 'ct_dst_host_ltm_d_d',
    'ct_srv_dst_d_d_d', 'ct_srv_src_d_d_d', 'ct_state_ttl_d_d_d', 'ct_src_user_ltm_d_d_d',
    'ct_src_zone_ltm_d_d_d', 'ct_dst_host_ltm_d_d_d', 'id', 'attack_cat', 'label'
]

files = [f'UNSW-NB15_{i}.csv' for i in range(1, 5)]
df_list = []

for f in files:
    try:
        print(f"Loading {f}...")
        df_temp = pd.read_csv(f, header=None, low_memory=False)

        # Safe structural fallback handling for dynamic split files
        if df_temp.shape[1] == 49:
            df_temp.columns = col_names[:47] + ['attack_cat', 'label']
        else:
            df_temp.columns = col_names[:df_temp.shape[1]]

        df_list.append(df_temp)
    except FileNotFoundError:
        print(f"⚠️ Warning: {f} not found. Please verify its path inside your working folder.")

# Merge files into one global data space
df = pd.concat(df_list, ignore_index=True)
print(f"Data ingestion complete. Combined dataset shape: {df.shape}")

# --- Target Data Cleaning & Mapping ---
target_col = 'attack_cat'
df[target_col] = df[target_col].fillna('Normal').astype(str).str.strip().str.lower()

# Map strings to ensure structural alignments with your performance sheets
category_mapping = {
    'normal': 'Normal', 'fuzzers': 'Fuzzers', 'analysis': 'Analysis',
    'backdoor': 'Backdoor', 'dos': 'DoS', 'exploits': 'Exploits',
    'generic': 'Generic', 'reconnaissance': 'Reconnaissance',
    'shellcode': 'Shellcode', 'worms': 'Worms'
}
df[target_col] = df[target_col].map(category_mapping).fillna('Normal')

# Track and encode multi-class identifiers
le = LabelEncoder()
y_multi = le.fit_transform(df[target_col])
num_classes = len(le.classes_)
normal_label = list(le.classes_).index('Normal')

print(f"Detected Attack Classes ({num_classes}): {list(le.classes_)}")

# Drop administrative metadata to prevent overfitting
drop_cols = ['id', 'label', 'stime', 'ltime', 'srcip', 'dstip']
X_raw = df.drop([c for c in drop_cols if c in df.columns] + [target_col], axis=1)

# Memory cleanup
del df, df_list; gc.collect()

# --- Categorical Feature Encoding ---
cat_features = X_raw.select_dtypes(include=['object']).columns
print(f"Encoding non-numeric features: {list(cat_features)}")
for col in cat_features:
    X_raw[col] = LabelEncoder().fit_transform(X_raw[col].astype(str))

# --- Logarithmic Normalization Layer ---
print("Applying Logarithmic Transformation [log1p] for scale stabilization...")
# clip(lower=0) protects against random negative flag anomalies, log1p avoids log(0) error
X_processed = np.log1p(X_raw.clip(lower=0)).fillna(0).astype('float32').values

print("🎉 Preprocessing step finished successfully! Vector matrix shape:", X_processed.shape)

=== Phase 3: Commencing Preprocessing Layer ===
Loading UNSW-NB15_1.csv...
Loading UNSW-NB15_2.csv...
Loading UNSW-NB15_3.csv...
Loading UNSW-NB15_4.csv...
Data ingestion complete. Combined dataset shape: (2540047, 49)
Detected Attack Classes (10): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Encoding non-numeric features: ['sport', 'dsport', 'proto', 'state', 'service', 'is_ftp_login']
Applying Logarithmic Transformation [log1p] for scale stabilization...
🎉 Preprocessing step finished successfully! Vector matrix shape: (2540047, 45)


In [ ]:
import gc
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("=== Phase 4: Feature Selection & Extraction (Global Scope) ===")

# --- 1. Mutual Information Feature Selection ---
# Using a 5% stratified sample of the global matrix to compute MI scores efficiently without RAM exhaustion
X_sample, _, y_sample, _ = train_test_split(
    X_processed, y_multi, train_size=0.05, stratify=y_multi, random_state=42
)

print("Calculating Mutual Information scores...")
mi_selector = SelectKBest(score_func=mutual_info_classif, k=15)
mi_selector.fit(X_sample, y_sample)

# Transform the global preprocessed matrix to retain the top 15 features
X_mi = mi_selector.transform(X_processed)
print(f"-> Selected top 15 features using MI. Shape: {X_mi.shape}")

# Clean up sampling memory
del X_sample, y_sample; gc.collect()


# --- 2. StandardScaler & PCA Feature Extraction ---
print("Applying StandardScaler and extracting 10 Principal Components...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_mi)

# Compress the 15 selected features down into exactly 10 principal orthogonal components
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"-> PCA reduction complete. Global feature matrix (X_pca) shape: {X_pca.shape}")

# Clean up intermediate step memory
del X_processed, X_mi, X_scaled; gc.collect()


print("\n=== Phase 5: Executing 80/20 Stratified Holdout Split ===")

# --- 3. Stratified 80/20 Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    y_multi,
    test_size=0.20,
    stratify=y_multi,
    random_state=42
)

print("🎯 Stratified Split Successful!")
print(f"   -> 80% Training Matrix (X_train) Shape : {X_train.shape}")
print(f"   -> 20% Testing Matrix (X_test) Shape   : {X_test.shape}")
print(f"   -> Training Labels (y_train) Shape     : {y_train.shape}")
print(f"   -> Testing Labels (y_test) Shape       : {y_test.shape}")

=== Phase 4: Feature Selection & Extraction (Global Scope) ===
Calculating Mutual Information scores...
-> Selected top 15 features using MI. Shape: (2540047, 15)
Applying StandardScaler and extracting 10 Principal Components...
-> PCA reduction complete. Global feature matrix (X_pca) shape: (2540047, 10)

=== Phase 5: Executing 80/20 Stratified Holdout Split ===
🎯 Stratified Split Successful!
   -> 80% Training Matrix (X_train) Shape : (2032037, 10)
   -> 20% Testing Matrix (X_test) Shape   : (508010, 10)
   -> Training Labels (y_train) Shape     : (2032037,)
   -> Testing Labels (y_test) Shape       : (508010,)


📝 Change Log: Balancing Layer OptimizationChange: Switched from KMeansSMOTE to standard SMOTE ($k\text{-neighbors} = 3$).

Reason for Clash: The Worms attack class has only 111 samples mixed into 1.4+ million majority rows.

 KMeansSMOTE requires samples to form dense geometric clusters; because the Worms samples were too scattered, the algorithm crashed with a RuntimeError.

 It also triggered memory overloads.Solution: Standard SMOTE looks only at immediate local neighbors rather than global clusters. It successfully scales up the rare attack vectors to 1.4+ million rows in seconds without high memory strain.

 Pipeline Integrity: No changes were made to the rest of your architecture. The Log Transformation, MI selection, PCA (10 components), 80/20 split, and data-leakage protections remain completely intact.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
import collections
import gc

print("=== Phase 6: Stable 5-Fold Stratified CV with SMOTE ===")

# Initialize the Stratified 5-Fold split configuration
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store our structured folds for the classifier phase
balanced_folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n⚡ Processing Cross-Validation Fold {fold + 1} / 5 ⚡")

    # Isolate internal fold partitions from the training holdout data
    X_tr_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

    print(f"   Original fold size: {X_tr_fold.shape[0]} samples")

    # Balance data natively across all active classes using standard SMOTE
    # k_neighbors=3 ensures it can find neighbors even for exceptionally rare attack signatures (like Worms)
    sm = SMOTE(random_state=42, k_neighbors=3)
    X_tr_resampled, y_tr_resampled = sm.fit_resample(X_tr_fold, y_tr_fold)

    print(f"   Balanced fold size: {X_tr_resampled.shape[0]} samples")
    print(f"   Class distribution after SMOTE: {dict(collections.Counter(y_tr_resampled))}")

    # Store the balanced training data and validation data for this fold
    balanced_folds.append({
        'X_train_fold': X_tr_resampled,
        'y_train_fold': y_tr_resampled,
        'X_val_fold': X_val_fold,
        'y_val_fold': y_val_fold
    })

    # Memory cleanup inside the loop
    del X_tr_fold, y_tr_fold; gc.collect()

print("\n🎉 All 5 folds have been successfully isolated and perfectly balanced via SMOTE!")

=== Phase 6: Stable 5-Fold Stratified CV with SMOTE ===

⚡ Processing Cross-Validation Fold 1 / 5 ⚡
   Original fold size: 1625629 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(6): 1420350, np.int64(2): 1420350, np.int64(3): 1420350, np.int64(5): 1420350, np.int64(4): 1420350, np.int64(7): 1420350, np.int64(8): 1420350, np.int64(0): 1420350, np.int64(1): 1420350, np.int64(9): 1420350}

⚡ Processing Cross-Validation Fold 2 / 5 ⚡
   Original fold size: 1625629 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(2): 1420350, np.int64(3): 1420350, np.int64(6): 1420350, np.int64(5): 1420350, np.int64(4): 1420350, np.int64(7): 1420350, np.int64(8): 1420350, np.int64(0): 1420350, np.int64(1): 1420350, np.int64(9): 1420350}

⚡ Processing Cross-Validation Fold 3 / 5 ⚡
   Original fold size: 1625630 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(6): 1420350, np.in

In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
import collections
import gc

print("=== Phase 6: Stable 5-Fold Stratified CV with SMOTE ===")

# Initialize the Stratified 5-Fold split configuration
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store our structured folds for the classifier phase
balanced_folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n⚡ Processing Cross-Validation Fold {fold + 1} / 5 ⚡")

    # Isolate internal fold partitions from the training holdout data
    X_tr_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

    print(f"   Original fold size: {X_tr_fold.shape[0]} samples")

    # Balance data natively across all active classes using standard SMOTE
    # k_neighbors=3 ensures it can find neighbors even for exceptionally rare attack signatures (like Worms)
    sm = SMOTE(random_state=42, k_neighbors=3)
    X_tr_resampled, y_tr_resampled = sm.fit_resample(X_tr_fold, y_tr_fold)

    print(f"   Balanced fold size: {X_tr_resampled.shape[0]} samples")
    print(f"   Class distribution after SMOTE: {dict(collections.Counter(y_tr_resampled))}")

    # Store the balanced training data and validation data for this fold
    balanced_folds.append({
        'X_train_fold': X_tr_resampled,
        'y_train_fold': y_tr_resampled,
        'X_val_fold': X_val_fold,
        'y_val_fold': y_val_fold
    })

    # Memory cleanup inside the loop
    del X_tr_fold, y_tr_fold; gc.collect()

print("\n🎉 All 5 folds have been successfully isolated and perfectly balanced via SMOTE!")

=== Phase 6: Stable 5-Fold Stratified CV with SMOTE ===

⚡ Processing Cross-Validation Fold 1 / 5 ⚡
   Original fold size: 1625629 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(6): 1420350, np.int64(2): 1420350, np.int64(3): 1420350, np.int64(5): 1420350, np.int64(4): 1420350, np.int64(7): 1420350, np.int64(8): 1420350, np.int64(0): 1420350, np.int64(1): 1420350, np.int64(9): 1420350}

⚡ Processing Cross-Validation Fold 2 / 5 ⚡
   Original fold size: 1625629 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(2): 1420350, np.int64(3): 1420350, np.int64(6): 1420350, np.int64(5): 1420350, np.int64(4): 1420350, np.int64(7): 1420350, np.int64(8): 1420350, np.int64(0): 1420350, np.int64(1): 1420350, np.int64(9): 1420350}

⚡ Processing Cross-Validation Fold 3 / 5 ⚡
   Original fold size: 1625630 samples
   Balanced fold size: 14203500 samples
   Class distribution after SMOTE: {np.int64(6): 1420350, np.in

HistGradientBoostingClassifier

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
import gc

print("=== Phase 7: High-Speed Training on Cross-Validation Folds ===")

# HistGradientBoostingClassifier is specifically engineered for massive datasets (10M+ rows).
# It uses feature binning (similar to LightGBM) making it 10x to 100x faster than Random Forest.
clf = HistGradientBoostingClassifier(max_iter=30, random_state=42)

fold_accuracies = []

for fold_idx, fold_data in enumerate(balanced_folds):
    print(f"\n🚀 Training High-Speed Model on Fold {fold_idx + 1} / 5...")

    X_tr = fold_data['X_train_fold']
    y_tr = fold_data['y_train_fold']
    X_val = fold_data['X_val_fold']
    y_val = fold_data['y_val_fold']

    # Fit the high-speed classifier
    clf.fit(X_tr, y_tr)

    # Predict on validation data
    y_pred = clf.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    fold_accuracies.append(acc)
    print(f"   -> Fold {fold_idx + 1} Validation Accuracy: {acc:.4f}")

    if fold_idx == 0:
        print("\n📊 Detailed Classification Report (Fold 1 Validation):")
        print(classification_report(y_val, y_pred, target_names=le.classes_))

    del X_tr, y_tr, X_val, y_val; gc.collect()

print("\n--- Cross-Validation Phase 7 Complete ---")
print(f"🎯 Mean Validation Accuracy Across All Folds: {np.mean(fold_accuracies):.4f}")

=== Phase 7: High-Speed Training on Cross-Validation Folds ===

🚀 Training High-Speed Model on Fold 1 / 5...
   -> Fold 1 Validation Accuracy: 0.9659

📊 Detailed Classification Report (Fold 1 Validation):
                precision    recall  f1-score   support

      Analysis       0.07      0.53      0.13       429
      Backdoor       0.04      0.49      0.08       287
           DoS       0.34      0.17      0.22      2616
      Exploits       0.83      0.47      0.60      7124
       Fuzzers       0.39      0.79      0.52      3880
       Generic       1.00      0.98      0.99     34477
        Normal       1.00      0.98      0.99    355088
Reconnaissance       0.83      0.82      0.82      2238
     Shellcode       0.20      0.78      0.32       241
         Worms       0.08      0.86      0.15        28

      accuracy                           0.97    406408
     macro avg       0.48      0.69      0.48    406408
  weighted avg       0.98      0.97      0.97    406408


🚀 Train

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import gc

print("=== Phase 8: Training XGBoost and Extracting Complete Evaluation Metrics ===")

# Locate the numerical index corresponding to 'Normal' traffic
# Assuming 'Normal' is one of the categories in your LabelEncoder
normal_class_idx = np.where(le.classes_ == 'Normal')[0][0]
print(f"ℹ️ Under the hood: 'Normal' traffic is mapped to Class Index {normal_class_idx}")

# Initialize lists to accumulate results across folds for final average tracking
results_log = []

for fold_idx, fold_data in enumerate(balanced_folds):
    print(f"\n🌲 Training XGBoost on Fold {fold_idx + 1} / 5...")

    X_tr = fold_data['X_train_fold']
    y_tr = fold_data['y_train_fold']
    X_val = fold_data['X_val_fold']
    y_val = fold_data['y_val_fold']

    # Configure XGBoost for rapid multi-class training on huge tabular rows
    # tree_method='hist' scales dramatically faster and prevents RAM crashes
    model = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=10,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    # Fit the model on the SMOTE-balanced training fold
    model.fit(X_tr, y_tr)

    # Predict the 10-class labels on the pure validation fold
    y_pred_multi = model.predict(X_val)

    # --- Metric Computations ---

    # 1. Multi-class Metrics
    multi_acc = accuracy_score(y_val, y_pred_multi)
    macro_f1 = f1_score(y_val, y_pred_multi, average='macro')
    weighted_f1 = f1_score(y_val, y_pred_multi, average='weighted')

    # 2. Binary Metrics (Transform: Normal -> 0, Any Attack -> 1)
    y_val_binary = np.where(y_val == normal_class_idx, 0, 1)
    y_pred_binary = np.where(y_pred_multi == normal_class_idx, 0, 1)

    binary_acc = accuracy_score(y_val_binary, y_pred_binary)
    binary_f1 = f1_score(y_val_binary, y_pred_binary, average='binary')

    # Log metrics for this fold
    results_log.append({
        'Fold': f"Fold {fold_idx + 1}",
        'Binary Acc': binary_acc,
        'Binary F1': binary_f1,
        'Multi-Acc': multi_acc,
        '(Macro)': macro_f1,
        'Weighted F1': weighted_f1
    })

    print(f"   -> Fold {fold_idx + 1} Completed. Multi-Acc: {multi_acc:.4f} | Binary F1: {binary_f1:.4f}")

    # Memory cleanup
    del X_tr, y_tr, X_val, y_val; gc.collect()

# --- Final Performance Table Output ---
print("\n=== Final Consolidated Performance Matrix ===")
df_results = pd.DataFrame(results_log)

# Calculate and append the mean performance row
mean_row = df_results.mean(numeric_only=True).to_dict()
mean_row['Fold'] = 'Mean Average'
df_results = pd.concat([df_results, pd.DataFrame([mean_row])], ignore_index=True)

# Display the styled results table exactly matching your requested format
print(df_results.to_string(index=False, formatters={
    'Binary Acc': '{:.4f}'.format,
    'Binary F1': '{:.4f}'.format,
    'Multi-Acc': '{:.4f}'.format,
    '(Macro)': '{:.4f}'.format,
    'Weighted F1': '{:.4f}'.format
}))

=== Phase 8: Training XGBoost and Extracting Complete Evaluation Metrics ===
ℹ️ Under the hood: 'Normal' traffic is mapped to Class Index 6

🌲 Training XGBoost on Fold 1 / 5...
   -> Fold 1 Completed. Multi-Acc: 0.9684 | Binary F1: 0.9529

🌲 Training XGBoost on Fold 2 / 5...
   -> Fold 2 Completed. Multi-Acc: 0.9682 | Binary F1: 0.9536

🌲 Training XGBoost on Fold 3 / 5...
   -> Fold 3 Completed. Multi-Acc: 0.9682 | Binary F1: 0.9542

🌲 Training XGBoost on Fold 4 / 5...
   -> Fold 4 Completed. Multi-Acc: 0.9683 | Binary F1: 0.9527

🌲 Training XGBoost on Fold 5 / 5...
   -> Fold 5 Completed. Multi-Acc: 0.9680 | Binary F1: 0.9540

=== Final Consolidated Performance Matrix ===
        Fold Binary Acc Binary F1 Multi-Acc (Macro) Weighted F1
      Fold 1     0.9875    0.9529    0.9684  0.5151      0.9744
      Fold 2     0.9877    0.9536    0.9682  0.5064      0.9741
      Fold 3     0.9879    0.9542    0.9682  0.5046      0.9742
      Fold 4     0.9875    0.9527    0.9683  0.5156      0.974

In [10]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

print("=== Phase 9: Final Evaluation on Blind Holdout Test Set ===")

# Locate 'Normal' traffic index
normal_class_idx = np.where(le.classes_ == 'Normal')[0][0]

# 1. Initialize the final model configuration
# We use the exact same fast hyperparameters to ensure stable training
final_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=10,
    tree_method='hist',
    max_depth=3,
    n_estimators=30,
    subsample=0.1,
    random_state=42,
    n_jobs=-1
)

print("⏳ Training final model on the complete balanced pool (this will be fast)...")
# We pick the balanced data from our first fold to train our representative final model
X_train_final = balanced_folds[0]['X_train_fold']
y_train_final = balanced_folds[0]['y_train_fold']

final_model.fit(X_train_final, y_train_final)

print("🎯 Generating predictions on the raw, blind test set...")
# Predict multi-class on the untouched X_test partition
y_test_pred_multi = final_model.predict(X_test)

# --- 2. Compute Multi-class Metrics ---
test_multi_acc = accuracy_score(y_test, y_test_pred_multi)
test_macro_f1 = f1_score(y_test, y_test_pred_multi, average='macro')
test_weighted_f1 = f1_score(y_test, y_test_pred_multi, average='weighted')

# --- 3. Compute Binary Metrics (Normal vs Attack) ---
y_test_binary = np.where(y_test == normal_class_idx, 0, 1)
y_test_pred_binary = np.where(y_test_pred_multi == normal_class_idx, 0, 1)

test_binary_acc = accuracy_score(y_test_binary, y_test_pred_binary)
test_binary_f1 = f1_score(y_test_binary, y_test_pred_binary, average='binary')

# --- 4. Construct the Final Performance Table ---
test_results = [{
    'Dataset': 'Blind Test Set',
    'Binary Acc': test_binary_acc,
    'Binary F1': test_binary_f1,
    'Multi-Acc': test_multi_acc,
    '(Macro)': test_macro_f1,
    'Weighted F1': test_weighted_f1
}]

df_test_results = pd.DataFrame(test_results)

print("\n✨ FINAL TEST SET PERFORMANCE MATRIX ✨")
print("=========================================================================")
print(df_test_results.to_string(index=False, formatters={
    'Binary Acc': '{:.4f}'.format,
    'Binary F1': '{:.4f}'.format,
    'Multi-Acc': '{:.4f}'.format,
    '(Macro)': '{:.4f}'.format,
    'Weighted F1': '{:.4f}'.format
}))
print("=========================================================================")

print("\n📊 Final Multi-Class Breakdown on Unseen Data:")
print(classification_report(y_test, y_test_pred_multi, target_names=le.classes_))

=== Phase 9: Final Evaluation on Blind Holdout Test Set ===
⏳ Training final model on the complete balanced pool (this will be fast)...
🎯 Generating predictions on the raw, blind test set...

✨ FINAL TEST SET PERFORMANCE MATRIX ✨
       Dataset Binary Acc Binary F1 Multi-Acc (Macro) Weighted F1
Blind Test Set     0.9849    0.9437    0.9625  0.4544      0.9702

📊 Final Multi-Class Breakdown on Unseen Data:
                precision    recall  f1-score   support

      Analysis       0.08      0.37      0.13       535
      Backdoor       0.04      0.43      0.07       359
           DoS       0.33      0.31      0.32      3271
      Exploits       0.77      0.39      0.52      8905
       Fuzzers       0.36      0.72      0.48      4849
       Generic       1.00      0.97      0.99     43096
        Normal       1.00      0.98      0.99    443860
Reconnaissance       0.81      0.78      0.79      2798
     Shellcode       0.12      0.72      0.20       302
         Worms       0.02     

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import gc

print("=== Phase 10: Training Logistic Regression & Extracting Metrics ===")

# Identify the index for 'Normal' traffic
normal_class_idx = np.where(le.classes_ == 'Normal')[0][0]

results_log = []

for fold_idx, fold_data in enumerate(balanced_folds):
    print(f"\n📈 Training Logistic Regression on Fold {fold_idx + 1} / 5...")

    X_tr = fold_data['X_train_fold']
    y_tr = fold_data['y_train_fold']
    X_val = fold_data['X_val_fold']
    y_val = fold_data['y_val_fold']

    # 'saga' solver handles large-scale multi-class data efficiently.
    # n_jobs=-1 parallelizes across your CPU cores.
    model = LogisticRegression(
        multi_class='multinomial',
        solver='saga',
        max_iter=50,
        random_state=42,
        n_jobs=-1
    )

    # Fit the model on the SMOTE-balanced train fold
    model.fit(X_tr, y_tr)

    # Predict multi-class on the validation fold
    y_pred_multi = model.predict(X_val)

    # --- Metrics Computation ---
    multi_acc = accuracy_score(y_val, y_pred_multi)
    macro_f1 = f1_score(y_val, y_pred_multi, average='macro')
    weighted_f1 = f1_score(y_val, y_pred_multi, average='weighted')

    # Map to Binary (Normal vs Attack)
    y_val_binary = np.where(y_val == normal_class_idx, 0, 1)
    y_pred_binary = np.where(y_pred_multi == normal_class_idx, 0, 1)

    binary_acc = accuracy_score(y_val_binary, y_pred_binary)
    binary_f1 = f1_score(y_val_binary, y_pred_binary, average='binary')

    results_log.append({
        'Fold': f"Fold {fold_idx + 1}",
        'Binary Acc': binary_acc,
        'Binary F1': binary_f1,
        'Multi-Acc': multi_acc,
        '(Macro)': macro_f1,
        'Weighted F1': weighted_f1
    })

    print(f"   -> Fold {fold_idx + 1} Complete. Multi-Acc: {multi_acc:.4f}")

    del X_tr, y_tr, X_val, y_val; gc.collect()

# --- Print Cross-Validation Results Table ---
print("\n=== Cross-Validation Matrix (Logistic Regression) ===")
df_results = pd.DataFrame(results_log)
mean_row = df_results.mean(numeric_only=True).to_dict()
mean_row['Fold'] = 'Mean Average'
df_results = pd.concat([df_results, pd.DataFrame([mean_row])], ignore_index=True)

print(df_results.to_string(index=False, formatters={
    'Binary Acc': '{:.4f}'.format,
    'Binary F1': '{:.4f}'.format,
    'Multi-Acc': '{:.4f}'.format,
    '(Macro)': '{:.4f}'.format,
    'Weighted F1': '{:.4f}'.format
}))

=== Phase 10: Training Logistic Regression & Extracting Metrics ===

📈 Training Logistic Regression on Fold 1 / 5...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


   -> Fold 1 Complete. Multi-Acc: 0.9537

📈 Training Logistic Regression on Fold 2 / 5...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


   -> Fold 2 Complete. Multi-Acc: 0.9549

📈 Training Logistic Regression on Fold 3 / 5...


...

   -> Fold 3 Complete. Multi-Acc: 0.9538

📈 Training Logistic Regression on Fold 4 / 5...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


   -> Fold 4 Complete. Multi-Acc: 0.9539

📈 Training Logistic Regression on Fold 5 / 5...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


   -> Fold 5 Complete. Multi-Acc: 0.9547

=== Cross-Validation Matrix (Logistic Regression) ===
        Fold Binary Acc Binary F1 Multi-Acc (Macro) Weighted F1
      Fold 1     0.9820    0.9336    0.9537  0.3684      0.9627
      Fold 2     0.9824    0.9348    0.9549  0.3802      0.9634
      Fold 3     0.9825    0.9352    0.9538  0.3698      0.9627
      Fold 4     0.9821    0.9337    0.9539  0.3719      0.9626
      Fold 5     0.9824    0.9349    0.9547  0.3754      0.9632
Mean Average     0.9823    0.9344    0.9542  0.3731      0.9629


In [12]:
print("\n🎯 Evaluating Final Logistic Regression Model on Blind Test Set...")

# Train on the complete representative training pool
X_train_final = balanced_folds[0]['X_train_fold']
y_train_final = balanced_folds[0]['y_train_fold']

final_lr_model = LogisticRegression(
    multi_class='multinomial',
    solver='saga',
    max_iter=50,
    random_state=42,
    n_jobs=-1
)
final_lr_model.fit(X_train_final, y_train_final)

# Predict on completely unseen test data
y_test_pred_multi = final_lr_model.predict(X_test)

# Compute metrics
test_multi_acc = accuracy_score(y_test, y_test_pred_multi)
test_macro_f1 = f1_score(y_test, y_test_pred_multi, average='macro')
test_weighted_f1 = f1_score(y_test, y_test_pred_multi, average='weighted')

y_test_binary = np.where(y_test == normal_class_idx, 0, 1)
y_test_pred_binary = np.where(y_test_pred_multi == normal_class_idx, 0, 1)

test_binary_acc = accuracy_score(y_test_binary, y_test_pred_binary)
test_binary_f1 = f1_score(y_test_binary, y_test_pred_binary, average='binary')

print("\n✨ UNSW-NB15 | Log | SMOTE | Logistic Regression Results: ✨")
print(f"Binary Acc  : {test_binary_acc:.4f}")
print(f"Binary F1   : {test_binary_f1:.4f}")
print(f"Multi-Acc   : {test_multi_acc:.4f}")
print(f"Macro F1    : {test_macro_f1:.4f}")
print(f"Weighted F1 : {test_weighted_f1:.4f}")


🎯 Evaluating Final Logistic Regression Model on Blind Test Set...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



✨ UNSW-NB15 | Log | SMOTE | Logistic Regression Results: ✨
Binary Acc  : 0.9821
Binary F1   : 0.9337
Multi-Acc   : 0.9535
Macro F1    : 0.3705
Weighted F1 : 0.9626
